# Merge Wars: GitHub AI Brain — Jupyter Demo

This notebook demonstrates the MCP + Haystack + Gemini agent on GitHub repositories:
- Fetch issues/PRs/commits via MCP
- Summarize and analyze with Gemini
- Optional: use the Streamlit UI for a richer experience


In [ ]:
# Setup
import subprocess
import sys
from pathlib import Path

working_directory = Path.cwd()
requirements_file = next(
    (path for path in (
        working_directory / "ai_mlc" / "requirements.txt",
        working_directory.parent / "requirements.txt",
        working_directory / "requirements.txt",
    ) if path.is_file()),
    None,
)
if requirements_file is None:
    raise FileNotFoundError("Could not find ai_mlc/requirements.txt from this notebook directory.")
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(requirements_file)], check=True)
from dotenv import load_dotenv  # noqa: E402
load_dotenv()

DEMO_REPO = "octocat/Hello-World"  # change to your repo if desired


In [ ]:
# Initialize the shared MCP agent
import sys
from pathlib import Path
from dotenv import load_dotenv

project_root = Path.cwd()
if project_root.name == "demo" and project_root.parent.name == "ai_mlc":
    project_root = project_root.parent.parent
elif project_root.name == "ai_mlc":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
load_dotenv(project_root / ".env")

from ai_mlc.github_agent.agent import agent_fetch, setup_mcp_agent  # noqa: E402
from ai_mlc.github_agent.llm_gemini import gemini_chat  # noqa: E402

agent = setup_mcp_agent()
demo_history = []


In [ ]:
# Basic tool calls
print("Issues:\n", agent_fetch(agent, f"Use list_issues for {DEMO_REPO}. Summarize the latest open issues.", history=demo_history))
print("\nPRs:\n", agent_fetch(agent, f"Use list_pull_requests for {DEMO_REPO}. Summarize open pull requests.", history=demo_history))
print("\nCommits:\n", agent_fetch(agent, f"Use list_commits for {DEMO_REPO}. Summarize the last 5 commits.", history=demo_history))


In [ ]:
# Error analysis with Gemini
issues_text = agent_fetch(agent, f"Use search_github_issues for {DEMO_REPO}. Return issue titles, labels, and links.", history=demo_history)
source_context = agent_fetch(agent, f"Use search_code to find source relevant to open issues in {DEMO_REPO}, then get_file_contents for the most relevant file.", history=demo_history)
analysis = gemini_chat(
    f"These are the open issues and source excerpts for {DEMO_REPO}:\n{issues_text}\n{source_context}\n\n"
    "Suggest an evidence-based fix. Provide fixes as:\n"
    "FILE: <file_path>\n<new_content>"
)
analysis


## Streamlit UI
Run this in a terminal for the web UI:

```bash
streamlit run ai_mlc/stream_app.py
```
